# Phase 2b -- restoration training

Trains `RestoreUNet` (`src/restore_model.py`) to turn a degraded photo into a
clean one. Pairs are generated on the fly: a clean crop from `data/clean_pool/`,
plus a fresh random degradation from `src/degrade.py`.

**Loss** = `1.0 * L1  +  0.2 * (1 - SSIM)  +  0.05 * VGG-perceptual`
  - L1: pixel accuracy (alone -> slightly washed out)
  - SSIM: local contrast / structure
  - perceptual: texture that *looks* right (frozen VGG16 feature distance)

**Metrics** (val set, each epoch): PSNR, SSIM, LPIPS. Best checkpoint saved by val PSNR.
Sample `[degraded | restored | clean]` grids saved to `outputs/restore_samples/`.

Starting hyperparameters (to tune from the first run's curves + output images):
batch 16, ~30 epochs, Adam lr 2e-4 constant, mixed precision (AMP) for the 6GB GPU.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("working dir:", Path.cwd())

import sys
sys.path.insert(0, "src")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from pytorch_msssim import ssim as ssim_fn
import lpips as lpips_pkg

from restore_dataset import build_restore_loaders
from restore_model import RestoreUNet

torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.benchmark = True   # fixed 256x256 input -> let cuDNN pick fast kernels

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

working dir: c:\Users\Shamik\Desktop\ImageQuality_Classifier\ImageQuality_Classifier
device: cuda


## 1. Data

In [2]:
BATCH_SIZE = 16   # 6GB card fits this in float32; drop to 8 if it OOMs

loaders = build_restore_loaders(
    ["data/clean_pool_small/coco", "data/clean_pool_small/div2k"],   # shrunk pool (<=800px) -- avoids GPU starvation
    batch_size=BATCH_SIZE,
    num_workers=0,
)
for split, ldr in loaders.items():
    print(f"{split:5s} {len(ldr.dataset):5d} images | {len(ldr):4d} batches")

train  4128 images |  258 batches
val     384 images |   24 batches
test    288 images |   18 batches


## 2. The composite loss

`PerceptualLoss` runs both images through a **frozen** pretrained VGG16 and
compares its intermediate feature maps (indices 3 / 8 / 15 = relu1_2 / relu2_2 /
relu3_3 -- low and mid-level texture, not high-level semantics). VGG expects
ImageNet-normalised input, so we normalise inside.

`criterion` returns the weighted total **and** the three raw term values, so we
can watch whether they stay balanced (see the "loss weights" discussion).

In [3]:
class PerceptualLoss(nn.Module):
    def __init__(self, layer_idx=(3, 8, 15)):
        super().__init__()
        vgg = torchvision.models.vgg16(
            weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1).features
        self.slices = nn.ModuleList()
        prev = 0
        for idx in layer_idx:
            self.slices.append(nn.Sequential(*[vgg[i] for i in range(prev, idx + 1)]))
            prev = idx + 1
        for p in self.parameters():
            p.requires_grad_(False)
        self.eval()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        x = (pred - self.mean) / self.std
        y = (target - self.mean) / self.std
        loss = 0.0
        for s in self.slices:               # feed forward, accumulate at each checkpoint
            x, y = s(x), s(y)
            loss = loss + F.l1_loss(x, y)
        return loss


perceptual = PerceptualLoss().to(device)

# LADDER STEP 2: pure-L1 run was stable but too timid (model corrects ~30%% of the
# damage, +0.3 dB). Add SSIM back at 0.1 (half the original 0.2) -- the term that
# punishes the smooth near-identity output L1 tolerates. Perceptual still off; add
# it next if SSIM stays stable at lr 1e-4 + eps 1e-8 + grad-clip + warmup.
W_L1, W_SSIM, W_PERC = 1.0, 0.1, 0.0      # <- the loss weights to tune

def criterion(pred, target):
    l1 = F.l1_loss(pred, target)
    # only pay for SSIM / VGG when their weight is non-zero
    ssim_term = (1.0 - ssim_fn(pred, target, data_range=1.0)) if W_SSIM else torch.zeros((), device=pred.device)
    perc = perceptual(pred, target) if W_PERC else torch.zeros((), device=pred.device)
    total = W_L1 * l1 + W_SSIM * ssim_term + W_PERC * perc
    return total, {"l1": l1.item(), "ssim": float(ssim_term), "perc": float(perc)}

## 3. Model, optimizer, mixed precision

In [4]:
model = RestoreUNet().to(device)
# The pure-L1 diagnostic (12 ep, lr 3e-5, eps 1e-4) trained stably but far too slowly --
# output collapsed to near-identity. Stability came from dropping SSIM/perceptual, NOT from
# the throttled optimizer, so put a normal one back: eps 1e-4 -> 1e-8 (default, no damping),
# lr 3e-5 -> 1e-4 (canonical Adam restoration LR). Keep grad-clip + warmup as insurance.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.99), eps=1e-8)
# linear LR warmup over the first 300 steps
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=300)

print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

1,433,219 parameters


## 4. Metrics + sample grid

PSNR (higher better, dB), SSIM (0-1, higher better), LPIPS (lower better).

In [5]:
lpips_fn = lpips_pkg.LPIPS(net="alex").to(device)
for p in lpips_fn.parameters():
    p.requires_grad_(False)


def psnr(pred, target):
    mse = ((pred - target) ** 2).mean(dim=[1, 2, 3]).clamp(min=1e-10)   # per image
    return (10 * torch.log10(1.0 / mse)).mean()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_p = tot_s = tot_l = 0.0
    n = 0
    for degraded, clean in loader:
        degraded, clean = degraded.to(device), clean.to(device)
        out = model(degraded)
        bs = degraded.size(0)
        tot_p += psnr(out, clean).item() * bs
        tot_s += ssim_fn(out, clean, data_range=1.0).item() * bs
        tot_l += lpips_fn(out * 2 - 1, clean * 2 - 1).mean().item() * bs
        n += bs
    return tot_p / n, tot_s / n, tot_l / n


@torch.no_grad()
def save_sample_grid(model, loader, tag, n_rows=4):
    model.eval()
    degraded, clean = next(iter(loader))           # val loader is unshuffled+seeded -> same images every call
    degraded, clean = degraded[:n_rows].to(device), clean[:n_rows].to(device)
    restored = model(degraded)
    rows = []
    for i in range(n_rows):
        trip = torch.cat([degraded[i], restored[i], clean[i]], dim=2)  # [deg | restored | clean]
        rows.append(trip.cpu())
    grid = torch.cat(rows, dim=1).clamp(0, 1).permute(1, 2, 0).numpy()
    out_dir = Path("outputs/restore_samples")
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.imsave(out_dir / f"{tag}.png", grid)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\Shamik\.conda\envs\imageQuality_Classifier\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Shamik\.conda\envs\imageQuality_Classifier\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\Shamik\.conda\envs\imageQuality_Classifier\Lib\site-packages\lpips\weights\v0.1\alex.pth


c:\Users\Shamik\.conda\envs\imageQuality_Classifier\Lib\site-packages\lpips\lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model

## 5. Training loop

Each epoch: train over all crops (AMP forward, scaled backward), then measure on
val. Save the checkpoint whenever val PSNR improves. Save a sample grid every 5
epochs so we can *see* progress, not just read numbers.

In [ ]:
NUM_EPOCHS = 40   # pure L1 at a real LR -- expect visible denoising + exposure fix by ~ep 20
CKPT_PATH = Path("models/restore_best.pt")
CKPT_PATH.parent.mkdir(exist_ok=True)

history = []
best_psnr = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    run = {"total": 0.0, "l1": 0.0, "ssim": 0.0, "perc": 0.0}
    nb = 0
    for degraded, clean in loaders["train"]:
        degraded, clean = degraded.to(device), clean.to(device)

        optimizer.zero_grad()
        out = model(degraded)                          # float32, no AMP
        loss, parts = criterion(out, clean)
        if not torch.isfinite(loss) or loss.item() > 5.0:   # NaN/inf OR anomalous spike (normal ~0.1)
            print(f"  epoch {epoch} step {nb}: bad loss {loss.item():.3f}, skipped")
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # tame gradient spikes
        optimizer.step()
        scheduler.step()

        run["total"] += loss.item()
        for k in ("l1", "ssim", "perc"):
            run[k] += parts[k]
        nb += 1
    for k in run:
        run[k] /= nb

    val_psnr, val_ssim, val_lpips = evaluate(model, loaders["val"])
    history.append({"epoch": epoch, "train_total": run["total"],
                    "train_l1": run["l1"], "train_ssim": run["ssim"], "train_perc": run["perc"],
                    "val_psnr": val_psnr, "val_ssim": val_ssim, "val_lpips": val_lpips})

    flag = ""
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(model.state_dict(), CKPT_PATH)
        flag = "  <- saved"

    print(f"epoch {epoch:2d} | loss {run['total']:.4f} "
          f"(l1 {run['l1']:.4f}  ssim {run['ssim']:.4f}  perc {run['perc']:.3f}) | "
          f"val PSNR {val_psnr:5.2f}  SSIM {val_ssim:.3f}  LPIPS {val_lpips:.3f}{flag}")

    if epoch == 1 or epoch % 5 == 0:
        save_sample_grid(model, loaders["val"], f"epoch_{epoch:02d}")

print(f"\nbest val PSNR {best_psnr:.2f} -> {CKPT_PATH}")

epoch  1 | loss 0.0830 (l1 0.0830  ssim 0.0000  perc 0.000) | val PSNR 21.10  SSIM 0.626  LPIPS 0.358  <- saved
epoch  2 | loss 0.0822 (l1 0.0822  ssim 0.0000  perc 0.000) | val PSNR 21.03  SSIM 0.630  LPIPS 0.356
epoch  3 | loss 0.0812 (l1 0.0812  ssim 0.0000  perc 0.000) | val PSNR 21.10  SSIM 0.631  LPIPS 0.356
epoch  4 | loss 0.0812 (l1 0.0812  ssim 0.0000  perc 0.000) | val PSNR 21.25  SSIM 0.633  LPIPS 0.356  <- saved
epoch  5 | loss 0.0776 (l1 0.0776  ssim 0.0000  perc 0.000) | val PSNR 21.07  SSIM 0.633  LPIPS 0.357
epoch  6 | loss 0.0784 (l1 0.0784  ssim 0.0000  perc 0.000) | val PSNR 21.33  SSIM 0.637  LPIPS 0.355  <- saved
epoch  7 | loss 0.0799 (l1 0.0799  ssim 0.0000  perc 0.000) | val PSNR 21.01  SSIM 0.638  LPIPS 0.357
epoch  8 | loss 0.0782 (l1 0.0782  ssim 0.0000  perc 0.000) | val PSNR 21.35  SSIM 0.639  LPIPS 0.355  <- saved
epoch  9 | loss 0.0761 (l1 0.0761  ssim 0.0000  perc 0.000) | val PSNR 20.15  SSIM 0.633  LPIPS 0.362
epoch 10 | loss 0.0777 (l1 0.0777  ssim 0.

In [ ]:
import csv

# `history` is a list of per-epoch dicts. Write it straight to CSV -- no pandas.
_fields = list(history[0].keys())
with open("models/restore_history.csv", "w", newline="") as _f:
    _w = csv.DictWriter(_f, fieldnames=_fields)
    _w.writeheader()
    _w.writerows(history)

def col(key):                     # pull one column out of the history list
    return [h[key] for h in history]

# last 10 epochs, like the old hist.tail(10)
print(f"{'ep':>3}  {'train_total':>11}  {'val_psnr':>8}  {'val_ssim':>8}  {'val_lpips':>9}")
for h in history[-10:]:
    print(f"{h['epoch']:3d}  {h['train_total']:11.4f}  {h['val_psnr']:8.2f}  "
          f"{h['val_ssim']:8.3f}  {h['val_lpips']:9.3f}")

## 6. Diagnostics

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(col("epoch"), col("train_total")); ax[0].set_title("train loss (total)")
ax[1].plot(col("epoch"), col("val_psnr"), color="tab:green"); ax[1].set_title("val PSNR (higher better)")
ax[2].plot(col("epoch"), col("val_ssim"), label="SSIM")
ax[2].plot(col("epoch"), col("val_lpips"), label="LPIPS"); ax[2].legend(); ax[2].set_title("val SSIM / LPIPS")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# the three loss terms over training -- are they staying balanced?
fig, ax = plt.subplots(figsize=(7, 4))
for k, lab in [("train_l1", "L1"), ("train_ssim", "1-SSIM"), ("train_perc", "perceptual (raw)")]:
    ax.plot(col("epoch"), col(k), label=lab)
ax.set_yscale("log"); ax.legend(); ax.grid(alpha=0.3)
ax.set_xlabel("epoch"); ax.set_title("raw loss terms (before weighting)")
plt.show()

## 7. Final look -- best checkpoint on validation

In [ ]:
best = RestoreUNet().to(device)
best.load_state_dict(torch.load(CKPT_PATH, weights_only=True))
best.eval()

vp, vs, vl = evaluate(best, loaders["val"])
print(f"best checkpoint | val PSNR {vp:.2f}  SSIM {vs:.3f}  LPIPS {vl:.3f}")

save_sample_grid(best, loaders["val"], "final")
plt.figure(figsize=(12, 10))
plt.imshow(plt.imread("outputs/restore_samples/final.png"))
plt.axis("off"); plt.title("[ degraded | restored | clean ]"); plt.show()

## 8. FINAL -- synthetic test set (run once)

Uses the **sealed test split** (`loaders["test"]`, ~6% of the clean pool, deterministic
degradation, never touched during training or tuning). Run this only after you've
finished tuning the loss weights.

Reports two things:
- **reference metrics** (PSNR / SSIM / LPIPS) -- we have the clean ground truth for
  synthetic images, so these are the informative numbers here.
- **no-reference metrics** (BRISQUE, NIQE lower-better; MUSIQ higher-better) -- the
  *same yardstick* you'll use on your 20 real photos. Validating them here, where the
  ground truth is visible, tells you whether to trust them on the real photos where
  it isn't. For each, we compare the score on the degraded input vs the restored
  output -- restoration should move it in the good direction.

In [ ]:
import pyiqa

# --- reference metrics on the sealed synthetic test split ---
tp, ts, tl = evaluate(best, loaders["test"])
print(f"TEST (synthetic, reference)   PSNR {tp:.2f}   SSIM {ts:.3f}   LPIPS {tl:.3f}")

# --- no-reference metrics: degraded vs restored ---
nr = {n: pyiqa.create_metric(n, device=device) for n in ["brisque", "niqe", "musiq"]}

@torch.no_grad()
def no_ref_scores(model, loader):
    model.eval()
    acc = {n: {"deg": 0.0, "res": 0.0} for n in nr}
    n_img = 0
    for degraded, _clean in loader:
        degraded = degraded.to(device)
        restored = model(degraded)
        for name, m in nr.items():
            acc[name]["deg"] += m(degraded).sum().item()
            acc[name]["res"] += m(restored).sum().item()
        n_img += degraded.size(0)
    return {n: {k: v / n_img for k, v in d.items()} for n, d in acc.items()}

scores = no_ref_scores(best, loaders["test"])
print("\nno-reference  (degraded -> restored):")
for name, d in scores.items():
    improved = (d["res"] < d["deg"]) == nr[name].lower_better
    print(f"  {name:8s} {d['deg']:6.2f}  ->  {d['res']:6.2f}   {'better' if improved else 'WORSE'}"
          f"   ({'lower' if nr[name].lower_better else 'higher'} is better)")

## 9. Next

- Read the curves + the sample grids. Blurry output -> raise `W_PERC` / `W_SSIM`;
  over-sharpened / haloed -> lower `W_PERC`; colour/brightness off -> raise `W_L1`.
  Change one weight, re-run, compare. (~3-5 rounds, each a full retrain.)
- Then: evaluate on the 20 real photos (no-reference metrics + classifier-recheck),
  and integrate the model into `src/enhance.py`, replacing the classical noise/blur/
  exposure/contrast steps. Redeploy.